# Phase 11: Free Agent Targets
Fetch current ESPN free agents, match to pipeline projections, and rank by projected 2026 ESPN points.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import difflib
from src.data_builder import load_config
from espn_api.baseball import League

cfg = load_config()
espn_cfg = cfg['espn_league']

league = League(
    league_id=espn_cfg['league_id'],
    year=espn_cfg['year'],
    espn_s2=espn_cfg['espn_s2'],
    swid=espn_cfg['swid'],
)
print(f"Connected to league {espn_cfg['league_id']} ({espn_cfg['year']})")

Connected to league 30294024 (2026)


In [2]:
# Fetch all free agents across all positions (deduplicated by name)
POSITIONS = ['C', '1B', '2B', '3B', 'SS', 'OF', 'SP', 'RP']

seen = set()
free_agents = []
for pos in POSITIONS:
    for p in league.free_agents(position=pos, size=200):
        if p.name not in seen:
            seen.add(p.name)
            free_agents.append({
                'espn_name': p.name,
                'espn_pos': p.position,
                'pro_team': p.proTeam,
                'pct_owned': p.percent_owned,
            })

fa_df = pd.DataFrame(free_agents)
print(f"Total free agents found: {len(fa_df)}")
fa_df.head(10)

Total free agents found: 1303


,espn_name,espn_pos,pro_team,pct_owned
0,Hunter Goodman,C,Col,72.49
1,Alejandro Kirk,C,Tor,48.41
2,Gabriel Moreno,C,Ari,27.71
3,Samuel Basallo,C,Bal,21.70
4,Carter Jensen,C,KC,20.53
5,J.T. Realmuto,C,Phi,19.15
6,Austin Wells,C,NYY,12.77
7,Ryan Jeffers,C,Min,10.60
8,Kyle Teel,C,ChW,9.46
9,Nick Fortes,C,TB,6.40


In [3]:
# Load pipeline rankings and build full_name lookup
bat = pd.read_csv('../outputs/draft_rankings_batters.csv')
pit = pd.read_csv('../outputs/draft_rankings_pitchers.csv')

bat['full_name'] = bat['first'] + ' ' + bat['last']
pit['full_name'] = pit['first'] + ' ' + pit['last']
all_rankings = pd.concat([bat, pit], ignore_index=True)

# Build name lookup dict (full_name -> row index for fast merge)
all_names = all_rankings['full_name'].tolist()
name_set = set(all_names)

def match_name(espn_name):
    """Exact match first; fall back to fuzzy match for accents, hyphens, suffixes."""
    if espn_name in name_set:
        return espn_name
    matches = difflib.get_close_matches(espn_name, all_names, n=1, cutoff=0.85)
    return matches[0] if matches else None

fa_df['matched_name'] = fa_df['espn_name'].apply(match_name)

matched = fa_df[fa_df['matched_name'].notna()].copy()
unmatched = fa_df[fa_df['matched_name'].isna()].copy()

print(f"Matched to pipeline: {len(matched)}")
print(f"Unmatched (rookie / not in 2025 data): {len(unmatched)}")

Matched to pipeline: 400
Unmatched (rookie / not in 2025 data): 903


In [4]:
# Join projections and display ranked free agents
proj_cols = ['full_name', 'last', 'first', 'primary_pos', 'team',
             'projected_pts_2026', 'PAR', 'tier', 'risk_flags', 'overall_rank']
proj = all_rankings[proj_cols].rename(columns={'full_name': 'matched_name'})

fa_ranked = matched.merge(proj, on='matched_name', how='left')
fa_ranked = fa_ranked.sort_values('projected_pts_2026', ascending=False).reset_index(drop=True)
fa_ranked.index += 1  # 1-based rank

print(f"Free agent targets — top 30 by projected 2026 pts\n")
display_cols = ['espn_name', 'primary_pos', 'pro_team', 'pct_owned',
                'projected_pts_2026', 'PAR', 'tier', 'risk_flags']
display(fa_ranked[display_cols].head(30).round(1))

Free agent targets — top 30 by projected 2026 pts



,espn_name,primary_pos,pro_team,pct_owned,projected_pts_2026,PAR,tier,risk_flags
1,Julio Rodriguez,CF,FA,0.1,402.2,196.5,1,NaN
2,Heliot Ramos,LF,SF,17.1,368.8,154.1,1,NaN
3,Jung Hoo Lee,CF,SF,35.4,353.0,147.3,1,NaN
4,Ben Brown,P,ChC,1.2,338.2,150.1,1,NaN
5,Max Scherzer,P,Tor,10.7,330.0,141.9,1,NaN
6,Jordan Westburg,3B,Bal,18.8,322.6,140.8,1,NaN
7,Zebby Matthews,P,Min,0.8,321.4,133.3,1,NaN
8,Jaison Chourio,CF,Cle,0.1,312.8,107.1,1,NaN
9,Jurickson Profar,LF,Atl,6.0,304.7,90.0,1,NaN
10,TJ Friedl,CF,Cin,16.3,303.9,98.2,1,BABIP_regression


In [5]:
# Batters only
BATTER_POS = {'C', '1B', '2B', '3B', 'SS', 'OF', 'LF', 'CF', 'RF', 'DH', 'UTIL'}
fa_batters = fa_ranked[fa_ranked['primary_pos'].isin(BATTER_POS)]
print(f"Free agent batters: {len(fa_batters)}")
display(fa_batters[display_cols].head(20).round(1))

Free agent batters: 204


,espn_name,primary_pos,pro_team,pct_owned,projected_pts_2026,PAR,tier,risk_flags
1,Julio Rodriguez,CF,FA,0.1,402.2,196.5,1,NaN
2,Heliot Ramos,LF,SF,17.1,368.8,154.1,1,NaN
3,Jung Hoo Lee,CF,SF,35.4,353.0,147.3,1,NaN
6,Jordan Westburg,3B,Bal,18.8,322.6,140.8,1,NaN
8,Jaison Chourio,CF,Cle,0.1,312.8,107.1,1,NaN
9,Jurickson Profar,LF,Atl,6.0,304.7,90.0,1,NaN
10,TJ Friedl,CF,Cin,16.3,303.9,98.2,1,BABIP_regression
14,Caleb Durbin,3B,Bos,36.9,295.3,113.5,1,NaN
16,Nolan Schanuel,1B,LAA,9.0,292.9,106.5,1,NaN
17,Trevor Story,SS,Bos,24.4,292.4,68.9,1,NaN


In [6]:
# Pitchers only
fa_pitchers = fa_ranked[fa_ranked['primary_pos'] == 'P']
print(f"Free agent pitchers: {len(fa_pitchers)}")
display(fa_pitchers[display_cols].head(20).round(1))

Free agent pitchers: 196


,espn_name,primary_pos,pro_team,pct_owned,projected_pts_2026,PAR,tier,risk_flags
4,Ben Brown,P,ChC,1.2,338.2,150.1,1,NaN
5,Max Scherzer,P,Tor,10.7,330.0,141.9,1,NaN
7,Zebby Matthews,P,Min,0.8,321.4,133.3,1,NaN
11,Yu Darvish,P,SD,0.1,299.9,111.9,1,NaN
12,Edwin Diaz,P,Hou,0.0,299.2,161.1,1,ERA_regression
13,Chad Patrick,P,Mil,8.4,295.9,107.8,1,NaN
15,Dustin May,P,StL,1.6,293.6,105.5,1,NaN
19,Tylor Megill,P,NYM,0.1,291.5,103.4,1,NaN
27,Jacob Lopez,P,Oak,1.0,277.9,89.8,1,NaN
28,Dean Kremer,P,Bal,1.1,277.7,89.6,1,NaN


In [7]:
# Unmatched names — audit for name-format gaps or missing players
print("Unmatched free agents (no 2025 pipeline data or name mismatch):")
print(unmatched[['espn_name', 'espn_pos', 'pro_team', 'pct_owned']]
      .sort_values('pct_owned', ascending=False)
      .to_string(index=False))

Unmatched free agents (no 2025 pipeline data or name mismatch):
                   espn_name espn_pos pro_team  pct_owned
              Kazuma Okamoto       3B      Tor      45.57
                Tatsuya Imai       SP      Hou      32.13
                 Roki Sasaki       SP      LAD      25.49
              Samuel Basallo        C      Bal      21.70
               Carter Jensen        C       KC      20.53
                 Joey Wiemer       RF      Wsh      18.37
              Erik Sabrowski       RP      Cle      17.89
                Joe Musgrove       SP       SD      17.51
               Kyle Harrison       SP      Mil      11.99
             Justin Crawford       CF      Phi       9.67
                Carson Benge       CF      NYM       9.58
                Rhett Lowder       SP      Cin       7.45
                 Jared Jones       SP      Pit       5.84
                Javier Assad       SP      ChC       5.51
                  Cody Ponce       SP      Tor       5.12
        